# Stage 4 — Revenue Growth Driver Analysis

## Objective

I want to understand why sales increased or decreased and what contributed most to those changes.

Using the Online Retail II dataset, I'll look at sales trends, customer activity, products and countries to identify the main drivers of revenue growth.

## Key Questions

1. How did monthly revenue change?
2. Did growth come from more orders or higher spending per order?
3. Which countries and products contributed most?
4. What role did customer purchasing behaviour play?
5. How did returns and cancellations affect the results?
6. What could the business do with these findings?

## Analysis Scope

I'll start with the 2009–2010 data, using a consistent definition of revenue, before extending the analysis to both years.

I'll check my calculations and focus on what the data actually shows, without assuming that every pattern explains why sales changed.

In [1]:
#### Load the Dataset

import pandas as pd
import numpy as np

# Load the 2009–2010 worksheet
file_path = "../data/raw/online_retail_II.xlsx"

df = pd.read_excel(
    file_path,
    sheet_name="Year 2009-2010",
    engine="openpyxl"
)

print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

display(df.head())


Dataset shape: (525461, 8)

Column names:
['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [2]:
#### Inspect Transaction Types

# Inspect transaction types

df["LineValue"] = df["Quantity"] * df["Price"]

checks = {
    "Positive sales": ((df["Quantity"] > 0) & (df["Price"] > 0)).sum(),
    "Negative quantities": (df["Quantity"] < 0).sum(),
    "Zero prices": (df["Price"] == 0).sum(),
    "Negative prices": (df["Price"] < 0).sum(),
    "Cancellation lines": df["Invoice"].astype(str).str.startswith("C").sum()
}

for category, count in checks.items():
    print(f"{category}: {count:,}")

print(f"\nTotal signed value: £{df['LineValue'].sum():,.2f}")


Positive sales: 511,566
Negative quantities: 12,326
Zero prices: 3,687
Negative prices: 3
Cancellation lines: 10,206

Total signed value: £9,539,484.63


In [3]:
#### Run the Transaction Checks

# Check transaction types

print("Positive sales:",
      ((df["Quantity"] > 0) & (df["Price"] > 0)).sum())

print("Negative quantities:",
      (df["Quantity"] < 0).sum())

print("Zero prices:",
      (df["Price"] == 0).sum())

print("Negative prices:",
      (df["Price"] < 0).sum())

print("Cancellation lines:",
      df["Invoice"].astype(str).str.startswith("C").sum())

print("Total signed value: £",
      round((df["Quantity"] * df["Price"]).sum(), 2))


Positive sales: 511566
Negative quantities: 12326
Zero prices: 3687
Negative prices: 3
Cancellation lines: 10206
Total signed value: £ 9539484.63


In [4]:
#### Reconcile Negative Quantities

# Negative quantities without cancellation prefix

negative_non_c = df[
    (df["Quantity"] < 0) &
    (~df["Invoice"].astype(str).str.startswith("C"))
]

print("Negative non-cancellation lines:", len(negative_non_c))

print("\nPrice breakdown:")
print("Zero price:", (negative_non_c["Price"] == 0).sum())
print("Positive price:", (negative_non_c["Price"] > 0).sum())
print("Negative price:", (negative_non_c["Price"] < 0).sum())

display(negative_non_c.head(10))


Negative non-cancellation lines: 2121

Price breakdown:
Zero price: 2121
Positive price: 0
Negative price: 0


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,LineValue
263,489464,21733,85123a mixed,-96,2009-12-01 10:52:00,0.0,NaN,United Kingdom,-0.0
283,489463,71477,short,-240,2009-12-01 10:52:00,0.0,NaN,United Kingdom,-0.0
284,489467,85123A,21733 mixed,-192,2009-12-01 10:53:00,0.0,NaN,United Kingdom,-0.0
470,489521,21646,NaN,-50,2009-12-01 11:44:00,0.0,NaN,United Kingdom,-0.0
3114,489655,20683,NaN,-44,2009-12-01 17:26:00,0.0,NaN,United Kingdom,-0.0
3162,489660,35956,lost,-1043,2009-12-01 17:43:00,0.0,NaN,United Kingdom,-0.0
3168,489663,35605A,damages,-117,2009-12-01 18:02:00,0.0,NaN,United Kingdom,-0.0
4296,489806,18010,NaN,-770,2009-12-02 12:42:00,0.0,NaN,United Kingdom,-0.0
4538,489820,21133,invcd as 84879?,-720,2009-12-02 13:23:00,0.0,NaN,United Kingdom,-0.0
4566,489821,85049G,NaN,-240,2009-12-02 13:25:00,0.0,NaN,United Kingdom,-0.0


In [5]:
#### Inspect Non-Merchandise Transactions

# Inspect special stock codes

special_codes = [
    "M", "POST", "C2", "D",
    "BANK CHARGES", "PADS", "TEST001"
]

special = df[
    df["StockCode"].astype(str).str.upper().isin(special_codes)
].copy()

summary = (
    special.groupby("StockCode")
    .agg(
        Records=("Invoice", "size"),
        Total_Value=("LineValue", "sum")
    )
    .reset_index()
)

display(summary)

print("Total special transaction value:",
      round(special["LineValue"].sum(), 2))


,StockCode,Records,Total_Value
0,BANK CHARGES,65,-28386.990
1,C2,138,6400.000
2,D,100,-7788.320
3,M,850,-14122.130
4,PADS,15,-36.586
5,POST,865,46092.360
6,TEST001,15,202.500
7,m,4,12.500


Total special transaction value: 2373.33


In [6]:
#### Lets Calculate Gross Merchandise Sales

# Step 8 — Gross merchandise sales

excluded_codes = {
    "M", "POST", "C2", "D",
    "BANK CHARGES", "PADS", "TEST001"
}

stock_codes = df["StockCode"].astype(str).str.strip().str.upper()

gross_sales = df[
    (df["Quantity"] > 0) &
    (df["Price"] > 0) &
    (~df["Invoice"].astype(str).str.startswith("C")) &
    (~stock_codes.isin(excluded_codes))
].copy()

print("Gross merchandise sales lines:", len(gross_sales))

print(
    "Gross merchandise sales: £"
    f"{gross_sales['LineValue'].sum():,.2f}"
)

print(
    "Unique sales invoices:",
    gross_sales["Invoice"].nunique()
)


Gross merchandise sales lines: 510065
Gross merchandise sales: £9,986,178.83
Unique sales invoices: 20616


In [7]:
#### Lets Calculate Merchandise Cancellations

# Step 9 — Merchandise cancellations

cancellations = df[
    df["Invoice"].astype(str).str.startswith("C") &
    (~stock_codes.isin(excluded_codes))
].copy()

print("Cancellation lines:", len(cancellations))

print(
    "Signed cancellation value: £"
    f"{cancellations['LineValue'].sum():,.2f}"
)

print(
    "Unique cancellation invoices:",
    cancellations["Invoice"].nunique()
)

print(
    "Positive-value cancellation lines:",
    (cancellations["LineValue"] > 0).sum()
)


Cancellation lines: 9662
Signed cancellation value: £-312,515.51
Unique cancellation invoices: 4162
Positive-value cancellation lines: 0


In [8]:
#####Calculate Net Merchandise Sales


# Step 10 — Calculate net merchandise sales

gross_revenue = gross_sales["LineValue"].sum()
cancellation_value = cancellations["LineValue"].sum()

net_revenue = gross_revenue + cancellation_value

cancellation_rate = (
    abs(cancellation_value) / gross_revenue * 100
)

print(f"Gross merchandise sales: £{gross_revenue:,.2f}")
print(f"Merchandise cancellations: £{cancellation_value:,.2f}")
print(f"Provisional net merchandise sales: £{net_revenue:,.2f}")
print(f"Cancellation value as % of gross sales: {cancellation_rate:.2f}%")


Gross merchandise sales: £9,986,178.83
Merchandise cancellations: £-312,515.51
Provisional net merchandise sales: £9,673,663.32
Cancellation value as % of gross sales: 3.13%


In [9]:
##### Reconcile Revenue

# Step 11 — Revenue reconciliation

total_value = df["LineValue"].sum()

included_indices = (
    gross_sales.index.union(cancellations.index)
)

other_transactions = df.loc[
    ~df.index.isin(included_indices)
].copy()

other_value = other_transactions["LineValue"].sum()

reconciliation = pd.DataFrame({
    "Category": [
        "Gross merchandise sales",
        "Merchandise cancellations",
        "Other transactions",
        "Reconciled total",
        "Original dataset total"
    ],
    "Value": [
        gross_revenue,
        cancellation_value,
        other_value,
        gross_revenue + cancellation_value + other_value,
        total_value
    ]
})

display(reconciliation)

difference = (
    gross_revenue +
    cancellation_value +
    other_value -
    total_value
)

print(f"Reconciliation difference: £{difference:,.2f}")
print(f"Other transaction lines: {len(other_transactions):,}")


,Category,Value
0,Gross merchandise sales,9986178.830
1,Merchandise cancellations,-312515.510
2,Other transactions,-134178.686
3,Reconciled total,9539484.634
4,Original dataset total,9539484.634


Reconciliation difference: £0.00
Other transaction lines: 5,734


In [10]:
##### Lets  Investigate Other Transactions

# Step 12 — Investigate other transactions

other_transactions["TransactionType"] = np.select(
    [
        (other_transactions["Quantity"] < 0) &
        (other_transactions["Price"] == 0),

        (other_transactions["Price"] < 0),

        (other_transactions["Quantity"] > 0) &
        (other_transactions["Price"] > 0),

        (other_transactions["Quantity"] < 0) &
        (other_transactions["Price"] > 0)
    ],
    [
        "Zero-price negative adjustment",
        "Negative-price transaction",
        "Positive-value excluded transaction",
        "Negative-quantity priced transaction"
    ],
    default="Other / Review"
)

summary = (
    other_transactions
    .groupby("TransactionType")
    .agg(
        Records=("Invoice", "size"),
        Total_Value=("LineValue", "sum")
    )
    .reset_index()
)

display(summary)

print("Total records:", summary["Records"].sum())
print("Total value: £", round(summary["Total_Value"].sum(), 2))


,TransactionType,Records,Total_Value
0,Negative-price transaction,3,-136552.020
1,Negative-quantity priced transaction,543,-317713.430
2,Other / Review,1566,0.000
3,Positive-value excluded transaction,1501,320086.764
4,Zero-price negative adjustment,2121,0.000


Total records: 5734
Total value: £ -134178.69


In [11]:
#### Step 13: Investigate the 543 Transactions

# Step 13 — Investigate excluded negative-quantity transactions

negative_priced = other_transactions[
    (other_transactions["Quantity"] < 0) &
    (other_transactions["Price"] > 0)
].copy()

negative_priced["StockCode_Normalised"] = (
    negative_priced["StockCode"]
    .astype(str)
    .str.strip()
    .str.upper()
)

summary_negative = (
    negative_priced
    .groupby("StockCode_Normalised")
    .agg(
        Records=("Invoice", "size"),
        Total_Value=("LineValue", "sum")
    )
    .sort_values("Total_Value")
)

display(summary_negative.head(15))

print("Total lines:", len(negative_priced))
print(f"Total value: £{negative_priced['LineValue'].sum():,.2f}")


,Records,Total_Value
StockCode_Normalised,,
M,293,-277101.71
BANK CHARGES,42,-28756.23
D,95,-8186.21
POST,103,-3385.18
C2,5,-225.00
PADS,1,-36.60
TEST001,4,-22.50


Total lines: 543
Total value: £-317,713.43


In [12]:
##### 3 Lets Validate Positive Exclusions

# Step 14 — Investigate positive-value excluded transactions

positive_excluded = other_transactions[
    (other_transactions["Quantity"] > 0) &
    (other_transactions["Price"] > 0)
].copy()

positive_excluded["StockCode_Normalised"] = (
    positive_excluded["StockCode"]
    .astype(str)
    .str.strip()
    .str.upper()
)

positive_summary = (
    positive_excluded
    .groupby("StockCode_Normalised")
    .agg(
        Records=("Invoice", "size"),
        Total_Value=("LineValue", "sum")
    )
    .sort_values("Total_Value", ascending=False)
)

display(positive_summary)

print("Total lines:", len(positive_excluded))
print(f"Total value: £{positive_excluded['LineValue'].sum():,.2f}")


,Records,Total_Value
StockCode_Normalised,,
M,560,262992.080
POST,759,49477.540
C2,131,6625.000
D,5,397.890
BANK CHARGES,23,369.240
TEST001,9,225.000
PADS,14,0.014


Total lines: 1501
Total value: £320,086.76


In [13]:
##### Validate the Revenue Classification


# Step 15 — Final revenue classification check

excluded_priced = other_transactions[
    (other_transactions["Price"] > 0) &
    (other_transactions["Quantity"] != 0)
].copy()

unexpected_codes = excluded_priced[
    ~excluded_priced["StockCode"]
    .astype(str)
    .str.strip()
    .str.upper()
    .isin(excluded_codes)
]

print("Excluded priced transaction lines:", len(excluded_priced))
print("Unexpected stock code lines:", len(unexpected_codes))

display(unexpected_codes.head(10))


Excluded priced transaction lines: 2044
Unexpected stock code lines: 0


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,LineValue,TransactionType


## Revenue Definition and Validation

Before analysing revenue growth, I reviewed the transaction data to separate merchandise sales from cancellations, adjustments and other charges.

For this analysis, I define:

- **Gross merchandise sales:** Transactions with positive quantities and prices, excluding cancellation invoices and selected special stock codes.
- **Merchandise cancellations:** Signed values from cancellation invoices, excluding the same special stock codes.
- **Net merchandise sales:** Gross merchandise sales plus signed merchandise cancellations.

I excluded `M`, `POST`, `C2`, `D`, `BANK CHARGES`, `PADS` and `TEST001` from merchandise revenue. These represent special transactions that I am treating separately from standard product sales.

### Results

| Metric | Value |
|---|---:|
| Gross merchandise sales | £9,986,178.83 |
| Merchandise cancellations | −£312,515.51 |
| Provisional net merchandise sales | £9,673,663.32 |
| Cancellation value as percentage of gross sales | 3.13% |

### Validation

I reconciled the transaction categories against the original dataset value of £9,539,484.63. The reconciliation difference was £0.00.

I also checked the excluded priced transactions and found no unexpected stock codes.

### Limitations

This is an analytical definition of merchandise revenue, not an audited financial figure. Some manual transactions may relate to merchandise, and cancellations are measured using their recorded transaction values rather than matched to original orders.

Customer IDs are not required for this revenue calculation, so transactions without an identified customer remain included where they meet the merchandise sales criteria.

In [14]:
#### Lets Calculate Monthly Revenue


# Step 17 — Monthly merchandise revenue

gross_monthly = gross_sales.copy()
cancel_monthly = cancellations.copy()

# Convert transaction dates into monthly periods
gross_monthly["Month"] = pd.to_datetime(
    gross_monthly["InvoiceDate"]
).dt.to_period("M")

cancel_monthly["Month"] = pd.to_datetime(
    cancel_monthly["InvoiceDate"]
).dt.to_period("M")

# Calculate monthly values
monthly_gross = gross_monthly.groupby("Month")["LineValue"].sum()

monthly_cancellations = cancel_monthly.groupby("Month")["LineValue"].sum()

monthly_revenue = pd.DataFrame({
    "Gross_Sales": monthly_gross,
    "Cancellations": monthly_cancellations
}).fillna(0).sort_index()

monthly_revenue["Net_Revenue"] = (
    monthly_revenue["Gross_Sales"] +
    monthly_revenue["Cancellations"]
)

display(monthly_revenue.round(2))

print(
    "\nTotal net merchandise revenue: £"
    f"{monthly_revenue['Net_Revenue'].sum():,.2f}"
)



,Gross_Sales,Cancellations,Net_Revenue
Month,,,
2009-12,819895.92,-19561.28,800334.64
2010-01,634157.33,-8866.83,625290.50
2010-02,546205.20,-12584.39,533620.81
2010-03,774370.16,-10892.36,763477.80
2010-04,653964.40,-10280.27,643684.13
2010-05,651151.17,-38308.63,612842.54
2010-06,705045.37,-26370.66,678674.71
2010-07,641637.38,-17313.81,624323.57
2010-08,682887.12,-12739.95,670147.17



Total net merchandise revenue: £9,673,663.32


In [15]:
#### Lets Calculate Monthly Revenue Growth


# Step 18 — Monthly revenue growth

monthly_revenue["Growth_Rate"] = (
    monthly_revenue["Net_Revenue"].pct_change() * 100
)

monthly_revenue["Revenue_Change"] = (
    monthly_revenue["Net_Revenue"].diff()
)

display(
    monthly_revenue[
        ["Net_Revenue", "Revenue_Change", "Growth_Rate"]
    ].round(2)
)


,Net_Revenue,Revenue_Change,Growth_Rate
Month,,,
2009-12,800334.64,NaN,NaN
2010-01,625290.50,-175044.14,-21.87
2010-02,533620.81,-91669.69,-14.66
2010-03,763477.80,229856.99,43.07
2010-04,643684.13,-119793.67,-15.69
2010-05,612842.54,-30841.59,-4.79
2010-06,678674.71,65832.17,10.74
2010-07,624323.57,-54351.14,-8.01
2010-08,670147.17,45823.60,7.34


In [16]:
### Lets Investigate September–November Growth

# Step 19 — September to November growth

september = monthly_revenue.loc[
    "2010-09", "Net_Revenue"
]

november = monthly_revenue.loc[
    "2010-11", "Net_Revenue"
]

growth_value = november - september

growth_percentage = (
    growth_value / september
) * 100

print(f"September revenue: £{september:,.2f}")
print(f"November revenue: £{november:,.2f}")
print(f"Revenue increase: £{growth_value:,.2f}")
print(f"Revenue growth: {growth_percentage:.2f}%")


September revenue: £850,570.99
November revenue: £1,415,900.22
Revenue increase: £565,329.23
Revenue growth: 66.46%


In [17]:
#### Lets Complete: 66.46% revenue growth

monthly_orders = gross_monthly.groupby("Month").agg(
    Gross_Revenue=("LineValue", "sum"),
    Orders=("Invoice", "nunique")
)

monthly_orders["AOV"] = (
    monthly_orders["Gross_Revenue"] /
    monthly_orders["Orders"]
)

display(
    monthly_orders.loc[
        ["2010-09", "2010-11"]
    ].round(2)
)



,Gross_Revenue,Orders,AOV
Month,,,
2010-09,878348.96,1789,490.97
2010-11,1457727.81,2720,535.93


In [18]:
##### Lets Decompose Revenue Growth

# Step 21 — Revenue growth decomposition

sep = monthly_orders.loc["2010-09"]
nov = monthly_orders.loc["2010-11"]

order_growth = nov["Orders"] - sep["Orders"]
aov_growth = nov["AOV"] - sep["AOV"]

# Attribute the interaction using a symmetric decomposition
volume_effect = order_growth * (
    sep["AOV"] + nov["AOV"]
) / 2

aov_effect = aov_growth * (
    sep["Orders"] + nov["Orders"]
) / 2

total_growth = (
    nov["Gross_Revenue"] - sep["Gross_Revenue"]
)

print(f"Order volume effect: £{volume_effect:,.2f}")
print(f"AOV effect: £{aov_effect:,.2f}")
print(f"Total explained growth: £{volume_effect + aov_effect:,.2f}")
print(f"Actual gross revenue growth: £{total_growth:,.2f}")
print(f"Reconciliation difference: £{volume_effect + aov_effect - total_growth:,.2f}")


Order volume effect: £478,022.59
AOV effect: £101,356.26
Total explained growth: £579,378.85
Actual gross revenue growth: £579,378.85
Reconciliation difference: £0.00


In [19]:
#### Lets Reconcile Gross Growth with Net Growth

# Step 22 — Reconcile gross and net revenue growth

sep_cancel = monthly_revenue.loc[
    "2010-09", "Cancellations"
]

nov_cancel = monthly_revenue.loc[
    "2010-11", "Cancellations"
]

cancellation_effect = nov_cancel - sep_cancel

net_growth = (
    volume_effect +
    aov_effect +
    cancellation_effect
)

actual_net_growth = (
    monthly_revenue.loc["2010-11", "Net_Revenue"] -
    monthly_revenue.loc["2010-09", "Net_Revenue"]
)

print(f"Order volume effect: £{volume_effect:,.2f}")
print(f"AOV effect: £{aov_effect:,.2f}")
print(f"Cancellation effect: £{cancellation_effect:,.2f}")
print(f"Explained net growth: £{net_growth:,.2f}")
print(f"Actual net growth: £{actual_net_growth:,.2f}")
print(f"Difference: £{net_growth - actual_net_growth:,.2f}")


Order volume effect: £478,022.59
AOV effect: £101,356.26
Cancellation effect: £-14,049.62
Explained net growth: £565,329.23
Actual net growth: £565,329.23
Difference: £0.00


In [20]:
###### LEts Identify Countries Driving Revenue Growth


# Step 23 — Revenue growth by country

country_data = pd.concat([
    gross_monthly,
    cancel_monthly
])

country_data = country_data[
    country_data["Month"].astype(str).isin(
        ["2010-09", "2010-11"]
    )
]

country_revenue = (
    country_data
    .groupby(["Country", "Month"])["LineValue"]
    .sum()
    .unstack(fill_value=0)
)

country_revenue.columns = (
    country_revenue.columns.astype(str)
)

country_revenue["Growth"] = (
    country_revenue["2010-11"] -
    country_revenue["2010-09"]
)

country_revenue = country_revenue.sort_values(
    "Growth", ascending=False
)

display(country_revenue.head(10).round(2))

print(
    "Total country growth: £"
    f"{country_revenue['Growth'].sum():,.2f}"
)


Month,2010-09,2010-11,Growth
Country,,,
United Kingdom,725267.55,1241417.21,516149.66
France,181.84,18090.87,17909.03
Australia,785.83,18197.82,17411.99
Germany,15844.94,27054.91,11209.97
Switzerland,2274.44,12440.73,10166.29
Norway,973.69,4629.32,3655.63
Belgium,1070.06,4593.28,3523.22
Denmark,1725.25,3858.36,2133.11
Portugal,2826.18,4746.72,1920.54


Total country growth: £565,329.23


In [21]:
#### Lets Check Countries with Declining Revenue

# Step 24 — Countries with declining revenue

declining_countries = country_revenue[
    country_revenue["Growth"] < 0
].sort_values("Growth")

display(declining_countries.round(2))

print(
    "Number of declining countries:",
    len(declining_countries)
)

print(
    "Combined negative contribution: £"
    f"{declining_countries['Growth'].sum():,.2f}"
)



Month,2010-09,2010-11,Growth
Country,,,
Netherlands,44722.14,33360.66,-11361.48
Channel Islands,5221.34,1248.40,-3972.94
Singapore,2784.82,0.00,-2784.82
Japan,1895.00,-204.00,-2099.00
Spain,3245.20,1794.75,-1450.45
Unspecified,1967.95,528.77,-1439.18
Sweden,2819.56,1446.01,-1373.55
EIRE,30740.18,29659.14,-1081.04
Cyprus,1005.53,397.60,-607.93


Number of declining countries: 13
Combined negative contribution: £-27,237.82


In [22]:
##### Identify Products Driving Revenue Growth

# Step 25 — Product-level revenue growth

product_data = pd.concat([
    gross_monthly,
    cancel_monthly
])

product_data = product_data[
    product_data["Month"].astype(str).isin(
        ["2010-09", "2010-11"]
    )
].copy()

# Group by stock code to avoid splitting products
# when descriptions differ
product_revenue = (
    product_data
    .groupby(["StockCode", "Month"])["LineValue"]
    .sum()
    .unstack(fill_value=0)
)

product_revenue.columns = (
    product_revenue.columns.astype(str)
)

product_revenue["Growth"] = (
    product_revenue["2010-11"] -
    product_revenue["2010-09"]
)

product_revenue = product_revenue.sort_values(
    "Growth", ascending=False
)

display(product_revenue.head(10).round(2))

print(
    "Total product growth: £"
    f"{product_revenue['Growth'].sum():,.2f}"
)


Month,2010-09,2010-11,Growth
StockCode,,,
22086,3015.07,28321.16,25306.09
84347,5083.63,25710.94,20627.31
22910,4331.87,14270.32,9938.45
DOT,6436.58,15289.92,8853.34
22865,1355.91,9618.58,8262.67
22837,0.00,8130.11,8130.11
22197,2242.38,10259.99,8017.61
22866,1589.22,9320.14,7730.92
84879,6841.21,14083.93,7242.72


Total product growth: £565,329.23


In [23]:
##### Lets Identify Product Descriptions

# Step 26 — Identify top product descriptions

top_codes = product_revenue.head(10).index

product_descriptions = (
    product_data[
        product_data["StockCode"].isin(top_codes)
    ]
    .groupby("StockCode")["Description"]
    .agg(
        lambda x: x.dropna().mode().iloc[0]
        if not x.dropna().mode().empty
        else "Unknown"
    )
)

top_products = product_revenue.head(10).copy()

top_products["Description"] = product_descriptions

display(
    top_products[
        ["Description", "2010-09", "2010-11", "Growth"]
    ].round(2)
)


Month,Description,2010-09,2010-11,Growth
StockCode,,,,
22086,PAPER CHAIN KIT 50'S CHRISTMAS,3015.07,28321.16,25306.09
84347,ROTATING SILVER ANGELS T-LIGHT HLDR,5083.63,25710.94,20627.31
22910,PAPER CHAIN KIT VINTAGE CHRISTMAS,4331.87,14270.32,9938.45
DOT,DOTCOM POSTAGE,6436.58,15289.92,8853.34
22865,HAND WARMER OWL DESIGN,1355.91,9618.58,8262.67
22837,HOT WATER BOTTLE BABUSHKA,0.00,8130.11,8130.11
22197,SMALL POPCORN HOLDER,2242.38,10259.99,8017.61
22866,HAND WARMER SCOTTY DOG DESIGN,1589.22,9320.14,7730.92
84879,ASSORTED COLOUR BIRD ORNAMENT,6841.21,14083.93,7242.72


In [24]:
### Investigate DOTCOM POSTAGE

# Step 27 — Investigate DOTCOM POSTAGE

dot_transactions = df[
    df["StockCode"].astype(str).str.strip().str.upper() == "DOT"
].copy()

print("DOT transaction lines:", len(dot_transactions))

print(
    "Total signed DOT value: £"
    f"{dot_transactions['LineValue'].sum():,.2f}"
)

print(
    "Positive DOT value: £"
    f"{dot_transactions.loc[dot_transactions['LineValue'] > 0, 'LineValue'].sum():,.2f}"
)

print(
    "Negative DOT value: £"
    f"{dot_transactions.loc[dot_transactions['LineValue'] < 0, 'LineValue'].sum():,.2f}"
)

display(
    dot_transactions[
        ["Invoice", "Description", "Quantity", "Price", "LineValue"]
    ].head(10)
)


DOT transaction lines: 736
Total signed DOT value: £116,401.99
Positive DOT value: £116,408.71
Negative DOT value: £-6.72


,Invoice,Description,Quantity,Price,LineValue
2379,489597,DOTCOM POSTAGE,1,647.19,647.19
2539,489600,DOTCOM POSTAGE,1,55.96,55.96
2551,489601,DOTCOM POSTAGE,1,68.39,68.39
2571,489602,DOTCOM POSTAGE,1,59.35,59.35
2619,489603,DOTCOM POSTAGE,1,42.39,42.39
2644,489604,DOTCOM POSTAGE,1,50.87,50.87
2686,489607,DOTCOM POSTAGE,1,76.30,76.30
2698,489609,DOTCOM POSTAGE,1,74.61,74.61
2771,489612,DOTCOM POSTAGE,1,87.39,87.39
2806,489614,DOTCOM POSTAGE,1,80.30,80.30


In [25]:
#### Correct the Revenue Definition

# Step 28 — Correct merchandise revenue classification

excluded_codes = {
    "M", "POST", "DOT", "C2", "D",
    "BANK CHARGES", "PADS", "TEST001"
}

stock_codes = (
    df["StockCode"]
    .astype(str)
    .str.strip()
    .str.upper()
)

is_cancellation = (
    df["Invoice"]
    .astype(str)
    .str.startswith("C")
)

# Rebuild gross merchandise sales
gross_sales = df[
    (df["Quantity"] > 0) &
    (df["Price"] > 0) &
    (~is_cancellation) &
    (~stock_codes.isin(excluded_codes))
].copy()

# Rebuild merchandise cancellations
cancellations = df[
    is_cancellation &
    (~stock_codes.isin(excluded_codes))
].copy()

gross_revenue = gross_sales["LineValue"].sum()
cancellation_value = cancellations["LineValue"].sum()
net_revenue = gross_revenue + cancellation_value

print(f"Corrected gross merchandise sales: £{gross_revenue:,.2f}")
print(f"Corrected merchandise cancellations: £{cancellation_value:,.2f}")
print(f"Corrected net merchandise sales: £{net_revenue:,.2f}")
print(f"Previous net merchandise sales: £9,673,663.32")
print(f"Revenue adjustment: £{net_revenue - 9673663.32:,.2f}")


Corrected gross merchandise sales: £9,869,770.12
Corrected merchandise cancellations: £-312,508.79
Corrected net merchandise sales: £9,557,261.33
Previous net merchandise sales: £9,673,663.32
Revenue adjustment: £-116,401.99


In [26]:
#### Rebuild Monthly Revenue

# Step 29 — Rebuild monthly revenue using corrected data

gross_monthly = gross_sales.copy()
cancel_monthly = cancellations.copy()

gross_monthly["Month"] = pd.to_datetime(
    gross_monthly["InvoiceDate"]
).dt.to_period("M")

cancel_monthly["Month"] = pd.to_datetime(
    cancel_monthly["InvoiceDate"]
).dt.to_period("M")

monthly_revenue = pd.DataFrame({
    "Gross_Sales": gross_monthly.groupby("Month")["LineValue"].sum(),
    "Cancellations": cancel_monthly.groupby("Month")["LineValue"].sum()
}).fillna(0).sort_index()

monthly_revenue["Net_Revenue"] = (
    monthly_revenue["Gross_Sales"] +
    monthly_revenue["Cancellations"]
)

monthly_revenue["Growth_Rate"] = (
    monthly_revenue["Net_Revenue"].pct_change() * 100
)

display(monthly_revenue.round(2))

print(
    "\nCorrected total net merchandise revenue: £"
    f"{monthly_revenue['Net_Revenue'].sum():,.2f}"
)

print(
    "Reconciliation difference: £"
    f"{monthly_revenue['Net_Revenue'].sum() - net_revenue:,.2f}"
)


,Gross_Sales,Cancellations,Net_Revenue,Growth_Rate
Month,,,,
2009-12,801321.34,-19561.28,781760.06,NaN
2010-01,622816.82,-8866.83,613949.99,-21.47
2010-02,539761.70,-12584.39,527177.31,-14.13
2010-03,764376.40,-10892.36,753484.04,42.93
2010-04,649178.25,-10280.27,638897.98,-15.21
2010-05,645739.00,-38308.63,607430.37,-4.93
2010-06,700115.15,-26370.66,673744.49,10.92
2010-07,634982.09,-17310.45,617671.64,-8.32
2010-08,676215.89,-12736.59,663479.30,7.42



Corrected total net merchandise revenue: £9,557,261.33
Reconciliation difference: £-0.00


In [27]:
#### Recalculate September–November Growth

# Step 30 — Corrected September–November growth

september = monthly_revenue.loc[
    "2010-09", "Net_Revenue"
]

november = monthly_revenue.loc[
    "2010-11", "Net_Revenue"
]

growth_value = november - september

growth_percentage = (
    growth_value / september
) * 100

print(f"September revenue: £{september:,.2f}")
print(f"November revenue: £{november:,.2f}")
print(f"Revenue increase: £{growth_value:,.2f}")
print(f"Revenue growth: {growth_percentage:.2f}%")



September revenue: £844,134.41
November revenue: £1,400,610.30
Revenue increase: £556,475.89
Revenue growth: 65.92%


In [28]:
#### Recalculate Order Volume and AOV

# Step 31 — Corrected order volume and AOV

monthly_orders = (
    gross_monthly
    .groupby("Month")
    .agg(
        Gross_Revenue=("LineValue", "sum"),
        Orders=("Invoice", "nunique")
    )
)

monthly_orders["AOV"] = (
    monthly_orders["Gross_Revenue"] /
    monthly_orders["Orders"]
)

sep = monthly_orders.loc["2010-09"]
nov = monthly_orders.loc["2010-11"]

print("September 2010")
print(f"Orders: {sep['Orders']:,.0f}")
print(f"AOV: £{sep['AOV']:,.2f}")

print("\nNovember 2010")
print(f"Orders: {nov['Orders']:,.0f}")
print(f"AOV: £{nov['AOV']:,.2f}")

print(
    "\nOrder growth: "
    f"{(nov['Orders'] / sep['Orders'] - 1) * 100:.2f}%"
)

print(
    "AOV growth: "
    f"{(nov['AOV'] / sep['AOV'] - 1) * 100:.2f}%"
)


September 2010
Orders: 1,789
AOV: £487.37

November 2010
Orders: 2,720
AOV: £530.31

Order growth: 52.04%
AOV growth: 8.81%


In [29]:
#### Recalculate Revenue Growth Contributions

# Step 32 — Corrected revenue growth decomposition

order_growth = nov["Orders"] - sep["Orders"]
aov_growth = nov["AOV"] - sep["AOV"]

# Symmetric decomposition
volume_effect = order_growth * (
    sep["AOV"] + nov["AOV"]
) / 2

aov_effect = aov_growth * (
    sep["Orders"] + nov["Orders"]
) / 2

gross_growth = (
    nov["Gross_Revenue"] - sep["Gross_Revenue"]
)

print(f"Order volume effect: £{volume_effect:,.2f}")
print(f"AOV effect: £{aov_effect:,.2f}")
print(f"Total explained growth: £{volume_effect + aov_effect:,.2f}")
print(f"Actual gross revenue growth: £{gross_growth:,.2f}")
print(f"Reconciliation difference: £{volume_effect + aov_effect - gross_growth:,.2f}")

print(
    f"\nOrder volume contribution: "
    f"{volume_effect / gross_growth * 100:.2f}%"
)

print(
    f"AOV contribution: "
    f"{aov_effect / gross_growth * 100:.2f}%"
)


Order volume effect: £473,731.07
AOV effect: £96,794.44
Total explained growth: £570,525.51
Actual gross revenue growth: £570,525.51
Reconciliation difference: £-0.00

Order volume contribution: 83.03%
AOV contribution: 16.97%


In [30]:
##### Reconcile Gross and Net Revenue Growth

# Step 33 — Corrected net revenue growth decomposition

sep_cancellations = monthly_revenue.loc[
    "2010-09", "Cancellations"
]

nov_cancellations = monthly_revenue.loc[
    "2010-11", "Cancellations"
]

cancellation_effect = (
    nov_cancellations - sep_cancellations
)

explained_net_growth = (
    volume_effect +
    aov_effect +
    cancellation_effect
)

actual_net_growth = (
    monthly_revenue.loc["2010-11", "Net_Revenue"] -
    monthly_revenue.loc["2010-09", "Net_Revenue"]
)

print(f"Order volume effect: £{volume_effect:,.2f}")
print(f"AOV effect: £{aov_effect:,.2f}")
print(f"Cancellation effect: £{cancellation_effect:,.2f}")
print(f"Explained net growth: £{explained_net_growth:,.2f}")
print(f"Actual net growth: £{actual_net_growth:,.2f}")
print(f"Reconciliation difference: £{explained_net_growth - actual_net_growth:,.2f}")


Order volume effect: £473,731.07
AOV effect: £96,794.44
Cancellation effect: £-14,049.62
Explained net growth: £556,475.89
Actual net growth: £556,475.89
Reconciliation difference: £0.00


In [31]:
#### Recalculate Country Contributions

# Step 34 — Corrected country revenue growth

country_data = pd.concat([
    gross_monthly,
    cancel_monthly
])

country_data = country_data[
    country_data["Month"].astype(str).isin(
        ["2010-09", "2010-11"]
    )
].copy()

country_revenue = (
    country_data
    .groupby(["Country", "Month"])["LineValue"]
    .sum()
    .unstack(fill_value=0)
)

country_revenue.columns = country_revenue.columns.astype(str)

country_revenue["Growth"] = (
    country_revenue["2010-11"] -
    country_revenue["2010-09"]
)

country_revenue = country_revenue.sort_values(
    "Growth", ascending=False
)

display(country_revenue.head(10).round(2))

print(
    f"\nTotal country growth: £"
    f"{country_revenue['Growth'].sum():,.2f}"
)

print(
    f"Reconciliation difference: £"
    f"{country_revenue['Growth'].sum() - actual_net_growth:,.2f}"
)


Month,2010-09,2010-11,Growth
Country,,,
United Kingdom,718830.97,1226127.29,507296.32
France,181.84,18090.87,17909.03
Australia,785.83,18197.82,17411.99
Germany,15844.94,27054.91,11209.97
Switzerland,2274.44,12440.73,10166.29
Norway,973.69,4629.32,3655.63
Belgium,1070.06,4593.28,3523.22
Denmark,1725.25,3858.36,2133.11
Portugal,2826.18,4746.72,1920.54



Total country growth: £556,475.89
Reconciliation difference: £0.00


In [32]:
### Recalculate Product Growth Contributions

# Step 35 — Corrected product revenue growth

product_data = pd.concat([
    gross_monthly,
    cancel_monthly
])

product_data = product_data[
    product_data["Month"].astype(str).isin(
        ["2010-09", "2010-11"]
    )
].copy()

product_revenue = (
    product_data
    .groupby(["StockCode", "Month"])["LineValue"]
    .sum()
    .unstack(fill_value=0)
)

product_revenue.columns = product_revenue.columns.astype(str)

product_revenue["Growth"] = (
    product_revenue["2010-11"] -
    product_revenue["2010-09"]
)

product_revenue = product_revenue.sort_values(
    "Growth", ascending=False
)

# Attach product descriptions for interpretation
product_descriptions = (
    gross_monthly
    .dropna(subset=["Description"])
    .drop_duplicates(subset=["StockCode"], keep="last")
    .set_index("StockCode")["Description"]
)

product_revenue["Description"] = (
    product_revenue.index.map(product_descriptions)
)

display(
    product_revenue[
        ["Description", "2010-09", "2010-11", "Growth"]
    ].head(10).round(2)
)

print(
    f"\nTotal product growth: £"
    f"{product_revenue['Growth'].sum():,.2f}"
)

print(
    f"Reconciliation difference: £"
    f"{product_revenue['Growth'].sum() - actual_net_growth:,.2f}"
)


Month,Description,2010-09,2010-11,Growth
StockCode,,,,
22086,PAPER CHAIN KIT 50'S CHRISTMAS,3015.07,28321.16,25306.09
84347,ROTATING SILVER ANGELS T-LIGHT HLDR,5083.63,25710.94,20627.31
22910,PAPER CHAIN KIT VINTAGE CHRISTMAS,4331.87,14270.32,9938.45
22865,HAND WARMER OWL DESIGN,1355.91,9618.58,8262.67
22837,HOT WATER BOTTLE BABUSHKA,0.00,8130.11,8130.11
22197,SMALL POPCORN HOLDER,2242.38,10259.99,8017.61
22866,HAND WARMER SCOTTY DOG DESIGN,1589.22,9320.14,7730.92
84879,ASSORTED COLOUR BIRD ORNAMENT,6841.21,14083.93,7242.72
22867,HAND WARMER BIRD DESIGN,1336.03,7930.57,6594.54



Total product growth: £556,475.89
Reconciliation difference: £0.00


In [33]:
### Measure Top 10 Product Concentration

# Step 36 — Top 10 product contribution

top_10_growth = product_revenue.head(10)["Growth"].sum()

top_10_share = (
    top_10_growth / actual_net_growth
) * 100

remaining_growth = actual_net_growth - top_10_growth

print(f"Top 10 product growth: £{top_10_growth:,.2f}")
print(f"Top 10 share of net growth: {top_10_share:.2f}%")
print(f"Remaining products' net growth: £{remaining_growth:,.2f}")

print(
    f"Reconciliation difference: £"
    f"{top_10_growth + remaining_growth - actual_net_growth:,.2f}"
)


Top 10 product growth: £108,403.78
Top 10 share of net growth: 19.48%
Remaining products' net growth: £448,072.11
Reconciliation difference: £0.00


In [34]:
#### Identify Products with Declining Revenue

# Step 37 — Products with declining revenue

declining_products = product_revenue[
    product_revenue["Growth"] < 0
].sort_values("Growth")

print("Top 10 declining products:")

display(
    declining_products[
        ["Description", "2010-09", "2010-11", "Growth"]
    ].head(10).round(2)
)

print(
    f"\nNumber of declining products: "
    f"{len(declining_products):,}"
)

print(
    f"Total revenue decline: £"
    f"{declining_products['Growth'].sum():,.2f}"
)


Top 10 declining products:


Month,Description,2010-09,2010-11,Growth
StockCode,,,,
21621,VINTAGE UNION JACK BUNTING,6505.78,1512.84,-4992.94
21622,VINTAGE UNION JACK CUSHION COVER,5033.63,628.48,-4405.15
21906,PHARMACIE FIRST AID TIN,4135.16,479.74,-3655.42
21624,VINTAGE UNION JACK DOORSTOP,4034.51,434.17,-3600.34
21770,OPEN CLOSED METAL SIGN,3369.60,0.00,-3369.60
22357,KINGS CHOICE BISCUIT TIN,3435.28,297.14,-3138.14
22359,GLASS JAR KINGS CHOICE,2343.80,41.33,-2302.47
22950,36 DOILIES VINTAGE CHRISTMAS,2384.59,89.49,-2295.10
22612,TEA COSY VINTAGE UNION JACK,2264.58,0.00,-2264.58



Number of declining products: 990
Total revenue decline: £-154,922.60


In [35]:
#### Reconcile Growing and Declining Products

# Step 38 — Growing vs declining products

growing_products = product_revenue[
    product_revenue["Growth"] > 0
]

unchanged_products = product_revenue[
    product_revenue["Growth"] == 0
]

positive_growth = growing_products["Growth"].sum()
negative_growth = declining_products["Growth"].sum()

reconciled_growth = positive_growth + negative_growth

print(f"Growing products: {len(growing_products):,}")
print(f"Declining products: {len(declining_products):,}")
print(f"Unchanged products: {len(unchanged_products):,}")

print(f"\nPositive growth: £{positive_growth:,.2f}")
print(f"Negative growth: £{negative_growth:,.2f}")
print(f"Net growth: £{reconciled_growth:,.2f}")

print(
    f"Reconciliation difference: £"
    f"{reconciled_growth - actual_net_growth:,.2f}"
)


Growing products: 2,203
Declining products: 990
Unchanged products: 61

Positive growth: £711,398.49
Negative growth: £-154,922.60
Net growth: £556,475.89
Reconciliation difference: £0.00


In [36]:
#### Quantify the Revenue Offset

# Step 39 — Product growth offset analysis

decline_offset_pct = (
    abs(negative_growth) / positive_growth
) * 100

retained_growth_pct = (
    reconciled_growth / positive_growth
) * 100

print(
    f"Positive product growth: £{positive_growth:,.2f}"
)

print(
    f"Revenue lost from declining products: "
    f"£{abs(negative_growth):,.2f}"
)

print(
    f"Growth offset by declining products: "
    f"{decline_offset_pct:.2f}%"
)

print(
    f"Positive growth retained: "
    f"{retained_growth_pct:.2f}%"
)

print(
    f"Percentage reconciliation: "
    f"{decline_offset_pct + retained_growth_pct:.2f}%"
)


Positive product growth: £711,398.49
Revenue lost from declining products: £154,922.60
Growth offset by declining products: 21.78%
Positive growth retained: 78.22%
Percentage reconciliation: 100.00%



## Revenue Growth Driver Analysis — Key Findings

### Analysis Period
September–November 2010, using the Year 2009–2010 worksheet.

Revenue represents merchandise transactions after excluding identified non-merchandise stock codes, including DOTCOM POSTAGE.

### 1. Revenue Performance
- September net revenue: £844,134.41
- November net revenue: £1,400,610.30
- Net revenue growth: £556,475.89 (+65.92%)

### 2. Revenue Growth Drivers
- Order volume contribution: £473,731.07
- Average order value contribution: £96,794.44
- Cancellation impact: -£14,049.62

Order volume accounted for 83.03% of gross merchandise revenue growth, compared with 16.97% from AOV.

### 3. Geographic Contributions
- United Kingdom: +£507,296.32
- UK share of total net growth: 91.16%
- Other countries combined: +£49,179.57

Growth was heavily concentrated in the UK market.

### 4. Product Performance
- Growing products: 2,203
- Declining products: 990
- Unchanged products: 61
- Positive product growth: £711,398.49
- Negative product growth: -£154,922.60
- Net product growth: £556,475.89

The top 10 growing products contributed £108,403.78, representing 19.48% of net revenue growth.

Declining products offset 21.78% of positive product growth.

### 5. Preliminary Interpretation
The increase in revenue was primarily associated with higher order volume.

Seasonal merchandise, including Christmas decorations and cold-weather products, featured prominently among growing products.

However, revenue declined for 990 products, indicating uneven performance across the catalogue.

### 6. Limitations
- The analysis covers one historical comparison period.
- Seasonal demand is suggested, not causally established.
- Changes in inventory availability and pricing were not independently verified.
- Order counts represent distinct invoices containing eligible gross merchandise sales.
- AOV is calculated using gross merchandise revenue, while overall revenue growth is reported net of cancellations.
- Non-merchandise exclusions depend on the defined stock-code classification.



## Business Recommendations

### Recommendation 1: Improve Seasonal Inventory Planning

**Evidence:** Net merchandise revenue increased by 65.92%
between September and November 2010, while order volume
increased by 52.04%. Christmas decorations and cold-weather
products were prominent among the highest-growing products.

**Recommendation:** Use historical monthly sales and
product-level demand patterns to improve inventory
forecasting ahead of the autumn and Christmas trading period.

**Expected benefit:** Improved product availability during
peak demand, with reduced risk of stockouts and excess
inventory.

**Measurement:** Monitor stockout rates, forecast accuracy,
inventory turnover and product-level sales performance.

**Limitation:** Inventory availability was not measured in
the dataset, so stockouts cannot be confirmed as a cause
of lost sales.



### Recommendation 2: Review Underperforming Products

**Evidence:** Revenue declined for 990 products between
September and November 2010. These declines totalled
£154,922.60, offsetting 21.78% of the gains generated
by growing products.

Several Vintage Union Jack products experienced
particularly large declines.

**Recommendation:** Introduce regular product-level
performance reviews to identify declining merchandise.

Investigate sales trends, stock availability, pricing
and seasonal demand before deciding whether to reorder,
promote, discount or discontinue individual products.

**Expected benefit:** Better inventory allocation,
reduced exposure to slow-moving merchandise and
improved product assortment decisions.

**Measurement:** Monitor product-level revenue trends,
sell-through rates, inventory turnover and gross margins.

**Limitation:** The dataset does not contain inventory
levels, procurement costs or profit margins. Therefore,
declining revenue alone cannot establish that a product
is unprofitable or should be discontinued.



### Recommendation 3: Evaluate International Growth Opportunities

**Evidence:** The United Kingdom contributed £507,296.32,
representing 91.16% of net merchandise revenue growth
between September and November 2010.

France, Australia, Germany and Switzerland also recorded
positive revenue growth.

**Recommendation:** Evaluate international markets with
positive sales momentum before committing resources
to expansion.

Compare country-level revenue trends, order frequency,
average order value and customer retention to identify
markets with sustainable growth potential.

Prioritise small-scale market tests rather than
immediate large-scale expansion.

**Expected benefit:** Identification of viable international
growth opportunities and potential diversification
of revenue sources.

**Measurement:** Monitor international revenue growth,
order frequency, repeat purchase rates and contribution
margins by country.

**Limitation:** The dataset does not provide shipping
costs, acquisition costs, market size or country-level
profitability. Positive revenue growth alone is
insufficient to justify expansion.



## Conclusion

Net merchandise revenue increased by £556,475.89
(65.92%) between September and November 2010.

Order volume was the main driver of gross revenue growth,
contributing 83.03%, while higher average order value
contributed 16.97%. Increased cancellations partially
offset these gains.

Geographically, the United Kingdom accounted for 91.16%
of net revenue growth.

At product level, 2,203 products recorded revenue
increases, while 990 experienced declines. Declining
products offset 21.78% of positive product growth.

Christmas decorations and cold-weather merchandise
featured prominently among the highest-growing products,
suggesting a possible seasonal influence.

Three business priorities emerge from the analysis:

1. Improve seasonal inventory forecasting.
2. Review underperforming product lines.
3. Evaluate international growth opportunities.

These recommendations should be tested using additional
inventory, profitability and customer data before
significant business decisions are made.

The findings describe historical associations and
revenue contributions, not proven causal relationships.


Validate the Notebook Before Reporting


### Revenue Definition and Methodology

The analysis uses the "Year 2009-2010" worksheet
from the UCI Online Retail II dataset.

Merchandise revenue is calculated using:

- Gross sales: Positive-quantity, positive-price
  merchandise transactions, excluding invoices
  marked as cancellations.
- Cancellations: Signed values from cancellation
  invoices for eligible merchandise.
- Net revenue: Gross merchandise sales plus
  merchandise cancellations.

The following StockCodes are excluded from
merchandise revenue:

M, POST, DOT, C2, D, BANK CHARGES, PADS, TEST001

These codes represent identified non-merchandise
transactions, charges, discounts, adjustments
or test records.

Corrected dataset totals:

- Gross merchandise sales: £9,869,770.12
- Merchandise cancellations: -£312,508.79
- Net merchandise revenue: £9,557,261.33

The primary growth comparison covers September
to November 2010.

December 2010 is a partial month and is not
treated as a complete monthly comparison.

Order volume is measured using distinct invoice
numbers containing eligible gross merchandise
sales.

Average Order Value (AOV) is calculated as
gross merchandise revenue divided by the
number of distinct gross merchandise orders.


In [37]:
### Final Revenue Reconciliation

# Step 46 — Final revenue consistency checks

import numpy as np

checks = {
    "Dataset revenue reconciliation": np.isclose(
        gross_revenue + cancellation_value,
        net_revenue
    ),

    "Monthly revenue reconciliation": np.isclose(
        monthly_revenue["Net_Revenue"].sum(),
        net_revenue
    ),

    "Growth decomposition": np.isclose(
        volume_effect + aov_effect + cancellation_effect,
        actual_net_growth
    ),

    "Country growth reconciliation": np.isclose(
        country_revenue["Growth"].sum(),
        actual_net_growth
    ),

    "Product growth reconciliation": np.isclose(
        product_revenue["Growth"].sum(),
        actual_net_growth
    ),

    "Product gains and declines": np.isclose(
        positive_growth + negative_growth,
        actual_net_growth
    )
}

for check, passed in checks.items():
    print(f"{check}: {'PASS' if passed else 'FAIL'}")

print(
    f"\nOverall result: "
    f"{'ALL CHECKS PASSED' if all(checks.values()) else 'REVIEW REQUIRED'}"
)


Dataset revenue reconciliation: PASS
Monthly revenue reconciliation: PASS
Growth decomposition: PASS
Country growth reconciliation: PASS
Product growth reconciliation: PASS
Product gains and declines: PASS

Overall result: ALL CHECKS PASSED
